# ✂️ Generación de Subconjunto de Validación (Calibración de Prompts)

## 🎯 Objetivo del Cuaderno
Este *notebook* tiene un propósito puramente logístico y de optimización de recursos. El procesamiento de imágenes y texto mediante Modelos de Lenguaje de Gran Escala (LLMs), como Google Gemini, conlleva un coste computacional y de consumo de API significativo. 

Para desarrollar, testear y calibrar la "Ingeniería de Prompts" sin iterar sobre el *dataset* completo, este script extrae una muestra representativa de 100 filas del conjunto de datos limpio. Este lote de prueba permite validar que la IA extrae correctamente el nivel de lujo, el estado de conservación y la calidad fotográfica, garantizando que el formato de salida sea el adecuado antes de escalar el proceso a miles de inmuebles.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

def dividir_parquet(archivo_entrada, directorio_salida, filas_por_archivo=100):
    """
    Divide un archivo Parquet en múltiples archivos con el número especificado de filas.
    
    Args:
        archivo_entrada (str): Ruta al archivo Parquet de entrada.
        directorio_salida (str): Directorio donde se guardarán los archivos resultantes.
        filas_por_archivo (int): Número de filas por archivo de salida.
    """
    # Crear directorio de salida si no existe
    Path(directorio_salida).mkdir(parents=True, exist_ok=True)
    
    # Leer el archivo Parquet
    tabla = pq.read_table(archivo_entrada)
    df = tabla.to_pandas()
    total_filas = len(df)
    
    print(f"Archivo original: {archivo_entrada}")
    print(f"Total de filas: {total_filas}")
    print(f"Dividiendo en archivos de {filas_por_archivo} filas...")
    
    # Calcular número de archivos necesarios
    num_archivos = (total_filas // filas_por_archivo) + (1 if total_filas % filas_por_archivo else 0)
    
    # Dividir y guardar
    for i in range(num_archivos):
        inicio = i * filas_por_archivo
        fin = inicio + filas_por_archivo
        lote = df.iloc[inicio:fin]
        
        nombre_archivo = f"{Path(archivo_entrada).stem}_parte_{i+1:03d}.parquet"
        ruta_salida = Path(directorio_salida) / nombre_archivo
        
        tabla_lote = pa.Table.from_pandas(lote)
        pq.write_table(tabla_lote, ruta_salida)
        
        print(f"Guardado {ruta_salida} con {len(lote)} filas")
    
    print(f"\nProceso completado! Se crearon {num_archivos} archivos en {directorio_salida}")

if __name__ == "__main__":
    # Configura tus rutas aquí
    ARCHIVO_PARQUET = "../DATOS/tecnocasa_datos_limpios.parquet"
    DIRECTORIO_SALIDA = "../DATOS/division100filas_tecnocasa_datoslimpios"
    
    # Opcional: cambiar el número de filas por archivo (por defecto 100)
    dividir_parquet(ARCHIVO_PARQUET, DIRECTORIO_SALIDA, filas_por_archivo=100)

Archivo original: C:/Users/mario/Downloads/tecnocasa_datos_limpios.parquet
Total de filas: 1015
Dividiendo en archivos de 100 filas...
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_001.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_002.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_003.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_004.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_005.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_006.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_007.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_008.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpios_parte_009.parquet con 100 filas
Guardado C:\Users\mario\Downloads\tecnocasa_datos_limpi

In [2]:
df = pd.read_parquet("../DATOS/division100filas_tecnocasa_datoslimpios/tecnocasa_datos_limpios_parte_010.parquet", engine = "fastparquet")
df

,Enlace,Precio,Dormitorios,Superficie,Num_baños,Descripción,Metro,Renfe,Bus,Farmacia,...,Latitud,Longitud,announcement_id,Precio_m2,Longitud_desc,Riqueza_desc,Polaridad_desc,Subjetividad_desc,Adjetivos_positivos,Urgencia_desc
900,https://www.tecnocasa.es/venta/piso/madrid/mad...,475000.0,3,131.0,2,Exclusivo Ático de Lujo: Un Refugio de Confort...,1,1,1,1,...,40.416705,-3.703582,1371,3625.954198,466.0,285.0,0.282143,0.571429,4.0,1.0
901,https://www.tecnocasa.es/venta/piso/madrid/mad...,305000.0,3,92.0,1,Tecnocasa Estudio Inmobiliario Amposta vende p...,1,0,1,1,...,40.416705,-3.703582,1372,3315.217391,47.0,41.0,0.000000,0.000000,0.0,0.0
902,https://www.tecnocasa.es/venta/piso/madrid/mad...,425000.0,2,78.0,1,Agencia inmobiliaria de MADRID - zona SALAMANC...,1,1,1,1,...,40.424977,-3.663700,1376,5448.717949,114.0,87.0,0.000000,0.000000,0.0,0.0
903,https://www.tecnocasa.es/venta/piso/madrid/alc...,167000.0,3,64.0,1,"Agencia Inmobiliaria de Alcalá de Henares, TEC...",0,1,1,1,...,40.481954,-3.363981,1381,2609.375000,237.0,163.0,0.000000,0.000000,1.0,0.0
904,https://www.tecnocasa.es/venta/piso/madrid/alc...,140000.0,3,66.0,1,Agencia inmobiliaria de TECNOCASA de ALCALÁ DE...,0,1,1,1,...,40.481954,-3.363981,1382,2121.212121,117.0,86.0,0.100000,0.400000,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,https://www.tecnocasa.es/venta/piso/madrid/vel...,160500.0,1,52.0,1,INMOBILIARIA TECNOCASA - ESTUDIO VELILLA - Ven...,0,0,1,1,...,40.367166,-3.487443,1560,3086.538462,131.0,103.0,0.000000,0.000000,1.0,0.0
996,https://www.tecnocasa.es/venta/piso/madrid/las...,330000.0,2,69.0,1,Inmobiliaria en Las Rozas De Madrid - Tecnocas...,0,1,1,1,...,40.493310,-3.875840,1569,4782.608696,264.0,164.0,0.000000,0.000000,1.0,0.0
997,https://www.tecnocasa.es/venta/piso/madrid/las...,330000.0,2,69.0,1,Inmobiliaria en Las Rozas De Madrid - Tecnocas...,0,1,1,1,...,40.493310,-3.875840,1570,4782.608696,265.0,165.0,0.000000,0.000000,1.0,0.0
998,https://www.tecnocasa.es/venta/piso/madrid/mad...,1499000.0,3,151.0,4,'Agencia inmobiliaria de MADRID - zona SALAMAN...,1,1,1,1,...,40.432694,-3.674509,1571,9927.152318,345.0,199.0,0.371429,0.680952,1.0,1.0
